# Manchester City Standalone Forecaster — Version 1

This notebook is the **user-facing operational version** of the project.

It does **not** require the historical Football-Data, FBref, ClubElo, FotMob, Understat, Transfermarkt, PCA, SVD, Bayesian-calibration, or dynamic-model notebooks to be rerun.

The package already contains:

- the frozen 2026/27 Bayesian parameters;
- the corrected 2026/27 league roster and opening priors;
- the frozen 2026/27 `Strength_Home` Poisson score model;
- the five Manchester City league results entered through **20 September 2026**;
- an append-only state file that preserves later results and forecasts.

The live Version 1 chain is:

$$
\boxed{
\text{entered EPL results}
\rightarrow
\text{Bayesian strength update}
\rightarrow
\Delta S + Home
\rightarrow
\lambda_{GF},\lambda_{GA}
\rightarrow
P(\text{score})
}
$$

Run the setup cell first. Before forecasting, enter completed results dated **strictly before the fixture date**. Save the City forecast before kickoff and before entering any results from that same calendar date. Afterward, enter the completed results and review the season report.

The model uses calendar dates without kickoff times. Even an earlier kickoff on the same day must wait until after that day's City forecast has been saved.

Keep `season_state.json` between sessions: it contains the results and issued forecasts. The bundled snapshot contains five results and **no saved forecasts**.


In [ ]:
from pathlib import Path
import pandas as pd

from city_online_forecast import CitySeasonForecaster

STATE_FILE = Path("season_state.json")

if not STATE_FILE.exists():
    raise FileNotFoundError(
        "season_state.json is missing. Keep it in the same folder as this notebook."
    )

online = CitySeasonForecaster.load(STATE_FILE)

display(online.summary())
display(online.strength_table().round(4))


## 1. Enter a completed Manchester City match

Edit the values below and set `ADD_CITY_RESULT = True`.

Goals are always entered **City first**, including away fixtures. For example, Crystal Palace 1–4 Man City means `venue="Away"`, `gf=4`, and `ga=1`.

**Before a forecast:** use this cell only for results dated strictly before the fixture you will predict. **After the forecasted match:** use it to record the completed score.

Save the City forecast before entering any results from its calendar date. Exact duplicate results are ignored; a conflicting result is rejected.


In [ ]:
ADD_CITY_RESULT = False

LATEST_DATE = "YYYY-MM-DD"
LATEST_OPPONENT = "Opponent"
LATEST_VENUE = "Home"   # "Home" or "Away"
LATEST_CITY_GOALS = 0
LATEST_OPPONENT_GOALS = 0

if ADD_CITY_RESULT:
    update = online.record_city_result(
        opponent=LATEST_OPPONENT,
        venue=LATEST_VENUE,
        date=LATEST_DATE,
        gf=LATEST_CITY_GOALS,
        ga=LATEST_OPPONENT_GOALS,
    )
    display(update)
else:
    print("No City result added. Set ADD_CITY_RESULT = True when ready.")


## 2. Enter other completed EPL matches

The Bayesian model updates strengths across the league. Adding other clubs' results gives the next forecast fuller information about its opponent.

Scores in this table use **home-team first** order. Several completed matches can be entered together.

**Before forecasting a City fixture dated D, add only results dated before D.** Save that City forecast before kickoff and before entering any results dated D. The model does not use kickoff times, so this rule also applies to earlier matches on the same day.

If same-day results have already been entered, the current model cannot issue a forecast for that date. Keep the recorded results and resume with a later fixture.


In [ ]:
OTHER_EPL_RESULTS = [
    # Template only: replace with an actual completed match before uncommenting.
    # {"Date": "YYYY-MM-DD", "HomeTeam": "Arsenal", "AwayTeam": "Chelsea", "FTHG": 2, "FTAG": 1},
]

if OTHER_EPL_RESULTS:
    league_update = online.record_results(pd.DataFrame(OTHER_EPL_RESULTS))
    display(league_update)
else:
    print("No additional EPL results added.")


## 3. Forecast the next Manchester City match

Enter the fixture **before kickoff**. Its date must be later than every result date already saved. Predict before entering any results from the fixture's date.

Calling `predict()` saves the issued forecast. Later results preserve that record and update the strengths used for subsequent forecasts.

All displayed probability tables use **percentages**. In the score grid, rows are City goals and columns are opponent goals. The grid covers scores from 0 to 5; `P_Outside_0_5` reports the probability outside that range. Top 5 scores are selected from the larger score distribution.


In [ ]:
NEXT_DATE = None          # actual fixture date in "YYYY-MM-DD" format
NEXT_OPPONENT = None      # use a team name from the strength table
NEXT_VENUE = None         # "Home" or "Away"

if all(v is not None for v in [NEXT_DATE, NEXT_OPPONENT, NEXT_VENUE]):
    prediction = online.predict(
        opponent=NEXT_OPPONENT,
        venue=NEXT_VENUE,
        date=NEXT_DATE,
    )

    print("MATCH")
    display(prediction["Match"].round(4))

    print("WIN / DRAW / LOSS PROBABILITIES (%)")
    display(prediction["Outcome_Probabilities_pct"].round(2))

    print("TOP 5 EXACT SCORES (%)")
    display(prediction["Top5"].round(2))

    print("0–5 SCORE GRID (%) — rows: City; columns: opponent")
    display((prediction["Grid_0_5"] * 100).round(2))
else:
    print("Fill NEXT_DATE, NEXT_OPPONENT, and NEXT_VENUE to issue a forecast.")


## 4. Review saved forecasts and results

`forecast_history()` lists every saved forecast. The view below converts its numeric probability columns to percentages; the stored probabilities retain their original 0–1 scale.

`season_report(replay_missing=False)` shows City's results and available saved forecasts, including pending fixtures. The bundled five results have no saved forecasts, so their rows initially show `No_saved_forecast` and blank prediction fields.

Check `Prediction_Source` when assessing forecasts: `Saved_before_match_date` was saved on an earlier date; `Saved_same_day_time_unverified` has no verified kickoff timestamp; `Saved_replay` was saved after the match date. Saving a forecast on the same day does not by itself prove it was issued before kickoff.

For an explicitly retrospective view, use `online.season_report(replay_missing=True)`. Reconstructed entries are labeled `Replayed_now` and are not added to the issued-forecast archive.

Back up `season_state.json` when moving the project or replacing the package.


In [ ]:
print("CURRENT MODEL STATUS")
display(online.summary())

print("CURRENT STRENGTH TABLE")
display(online.strength_table().round(4))

print("SAVED FORECAST HISTORY — PROBABILITIES (%)")
history = online.forecast_history().copy()
probability_columns = [
    "P_Win", "P_Draw", "P_Loss",
    "Top5_Probability_Mass", "P_Outside_0_5",
]
present_probability_columns = [
    column for column in probability_columns if column in history.columns
]
if present_probability_columns:
    history[present_probability_columns] = (
        history[present_probability_columns] * 100
    ).round(2)
    history = history.rename(columns={
        column: f"{column}_pct" for column in present_probability_columns
    })
display(history)

print("CITY SEASON REPORT — RESULTS AND SAVED FORECASTS")
display(online.season_report(replay_missing=False))


## Operational note

This package is intentionally a **frozen Version 1 forecaster**.

The Poisson coefficients are not refitted during the season. New match results update the Bayesian league-strength state, which changes future $\Delta S$ values and therefore future score distributions.

That means this first deployed model can continue iterating through the season without rerunning the upstream research notebooks.

A future Version 2 can replace the frozen model snapshot while preserving the same idea of a small operational interface plus an evolving state file.
